# scDiffusion用 Pancreasデータ準備

このノートブックは、scDiffusionのVAE_trainとcell_trainを実行するための
pancreasデータセットを準備します。

## 実行手順
1. 全セルを順番に実行
2. 最後にadata_pancreas.h5adファイルが生成される
3. 遺伝子数27998、細胞型8種類のデータができる

In [ ]:
# 必要なライブラリのインポート
import scvelo
import numpy as np
import scipy.sparse
import os

In [ ]:
# データセットのダウンロードと読み込み
print("Downloading pancreas dataset...")
adata = scvelo.datasets.pancreas(file_path='data/Pancreas/endocrinogenesis_day15.h5ad')
print(f"Original data shape: {adata.shape}")
print(f"Original data type: {type(adata.X)}")
print(f"Original data dtype: {adata.X.dtype}")

In [ ]:
# データの基本情報確認
print("=== Dataset Information ===")
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print(f"Cell types: {list(adata.obs.clusters.unique())}")
print(f"Number of cell types: {len(adata.obs.clusters.unique())}")

In [ ]:
# データの前処理
print("Processing data for scDiffusion compatibility...")

# カラム名を標準化（clustersをcelltypeに変更）
adata.obs = adata.obs.rename(columns={"clusters": "celltype"})
print("Renamed 'clusters' column to 'celltype'")

# データ型をfloat64に統一（PyTorchとの互換性向上）
if scipy.sparse.issparse(adata.X):
    # スパース行列の場合
    adata.X = adata.X.astype(np.float64)
    print("Converted sparse matrix to float64")
else:
    # 密行列の場合
    adata.X = np.ascontiguousarray(adata.X, dtype=np.float64)
    print("Converted dense matrix to contiguous float64")

In [ ]:
# 最終的なデータ形式の確認
print("=== Final Data Format ===")
print(f"Data shape: {adata.shape}")
print(f"Data type: {type(adata.X)}")
print(f"Data dtype: {adata.X.dtype}")
print(f"Is sparse: {scipy.sparse.issparse(adata.X)}")
print(f"Cell types: {list(adata.obs.celltype.unique())}")
print(f"Number of cell types: {len(adata.obs.celltype.unique())}")

# toarray()のテスト（scDiffusionで使用される）
if scipy.sparse.issparse(adata.X):
    test_array = adata.X.toarray()
    print(f"toarray() result dtype: {test_array.dtype}")
    print(f"toarray() result contiguous: {test_array.flags.c_contiguous}")
    del test_array  # メモリ解放

In [ ]:
# データの保存
output_file = "adata_pancreas.h5ad"
print(f"Saving processed data to {output_file}...")
adata.write(output_file)
print("Data saved successfully!")

# ファイルサイズの確認
file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
print(f"File size: {file_size:.1f} MB")

In [ ]:
# 保存されたデータの読み込みテスト
print("Testing saved data...")
import scanpy as sc
test_adata = sc.read_h5ad(output_file)
print(f"Test read - Shape: {test_adata.shape}")
print(f"Test read - Cell types: {len(test_adata.obs.celltype.unique())}")
print(f"Test read - Data type: {test_adata.X.dtype}")
print("✓ Data loading test passed!")

## 完了！

データ準備が完了しました。以下のファイルが生成されています：

- `adata_pancreas.h5ad`: scDiffusion用に処理された pancreasデータセット
- 遺伝子数: 27,998
- 細胞型数: 8
- データ形式: float64, PyTorch互換

次のステップ：
1. VAE_train.py を実行
2. cell_train.py を実行

または train.sh を実行して全体のパイプラインを実行してください。